# Smart Contract HGNN: Step-by-Step Training Notebook

This notebook mirrors the project pipeline described in `README.md`, `PLAN.md`, `ARCHITECTURE.md`, and `Hypergraph_Analysis.tex`.

It is organized in two parts:

1. **Walk through one contract** from Steps 1-6:
   - extract AST / CFG / call graph
   - build the dependency graph
   - construct node sets
   - compute node features
   - build hyperedges and the incidence matrix
   - run one HGNN forward pass
2. **Run Step 7 training** with 3-fold cross-validation using either:
   - preprocessed folds from `data/processed_dataset.pt`, or
   - on-the-fly preprocessing through the existing pipeline

This notebook is intended to be educational first, then usable for experiments.

## 0. Setup

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Make the repo root importable whether the notebook is launched from
# the project root or from notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from src.extraction.ast_cfg import extract_all
from src.extraction.gdep import build_gdep
from src.hypergraph.nodeset import build_node_sets
from src.hypergraph.features import FEATURE_DIM, build_feature_matrix, get_feature_config
from src.hypergraph.hyperedges import build_hyperedges
from src.model.hgnn import HGNN
from src.evaluation.train import generate_cv_splits, process_contract_list, compute_metrics

print(f"FEATURE_DIM = {FEATURE_DIM}")
feature_config = get_feature_config()
feature_config

## 1. Pick One Contract For The Walkthrough

By default, the notebook takes the first reentrant contract from the aggregated benchmark.

In [ ]:
folds = generate_cv_splits(n_splits=3, random_state=42)

sample_sol_path = None
sample_label = None
for path, label in folds[0]["train"]:
    if label == 1:
        sample_sol_path = path
        sample_label = label
        break

print("Sample contract:", sample_sol_path)
print("Contract label:", sample_label, "(1 = vulnerable, 0 = safe)")

## 2. Step 1: AST, CFG, Call Graph Extraction

In [ ]:
sample = extract_all(sample_sol_path)
if sample is None:
    raise RuntimeError("Extraction failed. Check solc/slither setup and dataset availability.")

print("Functions:", len(sample["functions"]))
print("State vars:", len(sample["state_vars"]))
print("Call sites:", len(sample["call_sites"]))
print("Call graph nodes:", sample["G_call"].number_of_nodes())
print("Call graph edges:", sample["G_call"].number_of_edges())
print("CFG functions:", len(sample["cfg"]))

In [ ]:
pd.DataFrame(sample["functions"]).head(10)

In [ ]:
pd.DataFrame(sample["call_sites"]).head(10)

## 3. Step 2: Build The Data Dependency Graph $G_{dep}$

In [ ]:
G_dep = build_gdep(sample["cfg"], sample["call_sites"], sample["state_vars"])
print("G_dep nodes:", G_dep.number_of_nodes())
print("G_dep edges:", G_dep.number_of_edges())

gdep_edges = list(G_dep.edges())[:20]
pd.DataFrame(gdep_edges, columns=["source", "target"])

## 4. Step 3: Construct Node Sets $V_f$, $V_s$, $V_c$, and $V$

In [ ]:
node_sets = build_node_sets(sample["functions"], sample["state_vars"], sample["call_sites"])
V_f = node_sets["V_f"]
V_s = node_sets["V_s"]
V_c = node_sets["V_c"]
V = node_sets["V"]
node_index = node_sets["node_index"]

print("|V_f| =", len(V_f))
print("|V_s| =", len(V_s))
print("|V_c| =", len(V_c))
print("|V|   =", len(V))

pd.DataFrame({
    "V_f": pd.Series(V_f[:10]),
    "V_s": pd.Series(V_s[:10]),
    "V_c": pd.Series(V_c[:10]),
})

## 5. Step 4: Build The Feature Matrix $X$

In [ ]:
X = build_feature_matrix(
    V=node_sets["V"],
    V_f=node_sets["V_f"],
    V_s=node_sets["V_s"],
    V_c=node_sets["V_c"],
    functions=sample["functions"],
    state_vars=sample["state_vars"],
    call_sites=sample["call_sites"],
    cfg=sample["cfg"],
)

print("X shape:", X.shape)
print("First row:", X[0])
print("Non-zero entries in first 5 rows:")
for i in range(min(5, len(X))):
    nz = np.nonzero(X[i])[0].tolist()
    print(i, V[i], nz)

## 6. Step 5: Build Hyperedges $E$ And Incidence Matrix $H_{inc}$

In [ ]:
E, H_inc = build_hyperedges(
    V=node_sets["V"],
    V_c=node_sets["V_c"],
    V_s=node_sets["V_s"],
    node_index=node_sets["node_index"],
    G_call=sample["G_call"],
    G_dep=G_dep,
    call_sites=sample["call_sites"],
)

print("Number of hyperedges:", len(E))
print("H_inc shape:", H_inc.shape)
print("Hyperedge sizes:", [len(e) for e in E[:10]])

example_edges = []
for idx, e_c in enumerate(E[:5]):
    example_edges.append({
        "hyperedge_idx": idx,
        "members": sorted(e_c),
    })
pd.DataFrame(example_edges)

## 7. Step 6: Run One HGNN Forward Pass

This is not training yet. It only checks that one processed contract can be fed through the model and produces one prediction per hyperedge.

In [ ]:
torch.manual_seed(42)
model = HGNN(
    in_dim=FEATURE_DIM,
    hidden_dim=64,
    n_layers=2,
    use_layernorm=True,
).to(device)

X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
H_tensor = torch.tensor(H_inc, dtype=torch.float32, device=device)

with torch.no_grad():
    y_pred = model(X_tensor, H_tensor, E, node_index)

print("Prediction tensor shape:", tuple(y_pred.shape))
pd.DataFrame(y_pred.cpu().numpy(), columns=["prob_safe", "prob_vuln"]).head(10)

## 8. Load Fold Data For Training

Preferred path:

- run `uv run python scripts/preprocess_dataset.py --output data/processed_dataset.pt`
- then load `data/processed_dataset.pt`

Fallback path:

- generate the folds and process contracts on the fly through the project pipeline

In [ ]:
PREPROCESSED_PATH = PROJECT_ROOT / "data" / "processed_dataset.pt"
USE_PREPROCESSED = PREPROCESSED_PATH.exists()

if USE_PREPROCESSED:
    saved = torch.load(PREPROCESSED_PATH, weights_only=False)
    fold_data = saved["fold_data"]
    metadata = saved.get("metadata", {})
    print("Loaded preprocessed data:")
    print(metadata)
else:
    print("No preprocessed file found. Processing folds from scratch.")
    fold_data = []
    for fold_idx, fold in enumerate(folds):
        print(f"Processing fold {fold_idx + 1}...")
        train_data = process_contract_list(fold["train"])
        val_data = process_contract_list(fold["val"])
        fold_data.append({"train": train_data, "val": val_data})

for i, fold in enumerate(fold_data):
    print(f"Fold {i + 1}: {len(fold['train'])} train, {len(fold['val'])} val")

## 9. Step 7: Training Configuration

In [ ]:
HIDDEN_DIM = 64
N_LAYERS = 2
LR = 1e-3
EPOCHS = 50
DROPOUT = 0.0
USE_LAYERNORM = True
SEEDS = [42, 0, 1, 2, 3]
CLASS_WEIGHTS = torch.tensor([1.0, 2.57], dtype=torch.float32, device=device)

print({
    "hidden_dim": HIDDEN_DIM,
    "n_layers": N_LAYERS,
    "lr": LR,
    "epochs": EPOCHS,
    "dropout": DROPOUT,
    "use_layernorm": USE_LAYERNORM,
    "seeds": SEEDS,
})

## 10. Training Helpers

In [ ]:
def train_epoch_gpu(model, optimizer, loss_fn, train_data, device):
    model.train()
    total_loss = 0.0
    total_edges = 0

    for contract in train_data:
        X = torch.tensor(contract["X"], dtype=torch.float32, device=device)
        H_inc = torch.tensor(contract["H_inc"], dtype=torch.float32, device=device)
        E = contract["E"]
        node_index = contract["node_index"]
        label = contract["label"]
        n_edges = contract["n_hyperedges"]

        labels = torch.full((n_edges,), label, dtype=torch.long, device=device)
        logits = model.forward_logits(X, H_inc, E, node_index)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * n_edges
        total_edges += n_edges

    return total_loss / max(total_edges, 1)


def evaluate_gpu(model, val_data, device):
    model.eval()
    all_preds = []
    all_labels = []
    predictions = []

    with torch.no_grad():
        for contract in val_data:
            X = torch.tensor(contract["X"], dtype=torch.float32, device=device)
            H_inc = torch.tensor(contract["H_inc"], dtype=torch.float32, device=device)
            E = contract["E"]
            node_index = contract["node_index"]
            label = contract["label"]
            n_edges = contract["n_hyperedges"]

            y_pred = model(X, H_inc, E, node_index)
            pred_labels = y_pred.argmax(dim=1).cpu().tolist()
            true_labels = [label] * n_edges

            all_preds.extend(pred_labels)
            all_labels.extend(true_labels)

            for j, (pred, prob) in enumerate(zip(pred_labels, y_pred.cpu().tolist())):
                predictions.append({
                    "sol_path": contract["sol_path"],
                    "hyperedge_idx": j,
                    "true_label": label,
                    "pred_label": pred,
                    "prob_safe": prob[0],
                    "prob_vuln": prob[1],
                })

    return compute_metrics(all_preds, all_labels, predictions)


def save_predictions_csv(predictions, path):
    df = pd.DataFrame(predictions)
    if not df.empty:
        df.to_csv(path, index=False)

## 11. Train Across Seeds And Folds

This cell saves:

- `checkpoints/hgnn_fold{N}_seed{S}.pt`
- `results/fold{N}_seed{S}_predictions.csv`
- aggregated summary tables used below

In [ ]:
results_dir = PROJECT_ROOT / "results"
checkpoints_dir = PROJECT_ROOT / "checkpoints"
results_dir.mkdir(exist_ok=True)
checkpoints_dir.mkdir(exist_ok=True)

all_results = []
metric_names = ["precision", "recall", "f1", "fnr", "fpr", "accuracy"]

for seed in SEEDS:
    print(f"\n{'=' * 72}")
    print(f"Seed {seed}")
    print(f"{'=' * 72}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    seed_fold_metrics = []

    for fold_idx, fold in enumerate(fold_data):
        train_data = fold["train"]
        val_data = fold["val"]
        print(f"\nFold {fold_idx + 1} | train={len(train_data)} | val={len(val_data)}")

        model = HGNN(
            in_dim=FEATURE_DIM,
            hidden_dim=HIDDEN_DIM,
            n_layers=N_LAYERS,
            use_layernorm=USE_LAYERNORM,
            dropout=DROPOUT,
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

        best_f1 = -1.0
        best_state = None
        loss_history = []

        for epoch in range(EPOCHS):
            avg_loss = train_epoch_gpu(model, optimizer, loss_fn, train_data, device)
            loss_history.append(avg_loss)

            if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == EPOCHS - 1:
                metrics = evaluate_gpu(model, val_data, device)
                print(
                    f"  Epoch {epoch + 1:3d} | loss={avg_loss:.4f} | "
                    f"P={metrics['precision']:.3f} "
                    f"R={metrics['recall']:.3f} "
                    f"F1={metrics['f1']:.3f} "
                    f"FNR={metrics['fnr']:.3f} "
                    f"FPR={metrics['fpr']:.3f}"
                )

                if metrics["f1"] >= best_f1:
                    best_f1 = metrics["f1"]
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state is not None:
            model.load_state_dict(best_state)
            ckpt_path = checkpoints_dir / f"hgnn_fold{fold_idx + 1}_seed{seed}.pt"
            torch.save(best_state, ckpt_path)

        final_metrics = evaluate_gpu(model, val_data, device)
        final_metrics["loss_history"] = loss_history
        seed_fold_metrics.append(final_metrics)

        pred_path = results_dir / f"fold{fold_idx + 1}_seed{seed}_predictions.csv"
        save_predictions_csv(final_metrics["predictions"], pred_path)

    all_results.append({"seed": seed, "fold_metrics": seed_fold_metrics})

print("\nTraining complete.")

## 12. Aggregate Results

In [ ]:
rows = []
all_values = {m: [] for m in metric_names}

for result in all_results:
    seed = result["seed"]
    for fold_idx, fm in enumerate(result["fold_metrics"], start=1):
        row = {"seed": seed, "fold": fold_idx}
        for m in metric_names:
            row[m] = fm[m]
            all_values[m].append(fm[m])
        rows.append(row)

detail_df = pd.DataFrame(rows)
detail_df

In [ ]:
summary_rows = []
for m in metric_names:
    vals = np.array(all_values[m], dtype=float)
    summary_rows.append({
        "metric": m,
        "mean": vals.mean(),
        "std": vals.std(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
detail_path = results_dir / "cv_detailed.csv"
summary_path = results_dir / "cv_summary.csv"

detail_df.to_csv(detail_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Saved:")
print(" -", detail_path)
print(" -", summary_path)

## 13. Optional: Plot Training Loss

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for fold_idx in range(3):
        ax = axes[fold_idx]
        for result in all_results:
            seed = result["seed"]
            loss_history = result["fold_metrics"][fold_idx].get("loss_history", [])
            if loss_history:
                ax.plot(loss_history, label=f"seed={seed}", alpha=0.8)
        ax.set_title(f"Fold {fold_idx + 1}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)

    axes[-1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed. Install it if you want plots.")